In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from agent_lab.model_hub import LLM_FAST

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=init_chat_model(**LLM_FAST),
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

#### stream_mode="updates"

在每个 Agent 步骤后发出一个事件。

每个 StreamPart 都是一个包含 type 、 ns 和 data 键的字典。使用 `chunk["type"]` 来确定流模式，使用 `chunk["data"]` 来访问 payload

In [9]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
    version="v2",
):
    print(chunk['type'])
    if chunk["type"] == "updates":
        print(f"data: {chunk['data']}")
        for step, data in chunk["data"].items():
            print(f"step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}\n")

updates
data: {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 286, 'total_tokens': 331, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 30}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': '661b344f-6663-4e9f-92e5-5f5fdbd87a94', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dc9ef-92b7-7ea2-a040-c437746a1555-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call_00_AzmjnRlGpokWBnLZeDRElDhd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 286, 'output_tokens': 45, 'total_tokens': 331, 'input_token_details': {'cache_read': 256}, 'output_token_details': {}})]}}
step: model
content: [{'type': '

#### stream_mode="messages"

在 LLM 生成 tokens 时进行流式传输

In [11]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="messages",
    version="v2",
):
    print(chunk['type'])
    if chunk["type"] == "messages":
        print(f"data: {chunk['data']}")
        # chunk['data'] is a list of (MessageChunk, dict) tuple, dict contains metadata.
        token, metadata = chunk["data"]
        print(f"node: {metadata['langgraph_node']}")
        print(f"content: {token.content_blocks}\n")

messages
data: (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'deepseek'}, id='lc_run--019dc9f3-64b7-7be1-8deb-92430364e280', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:68fd80eb-3be4-32bb-a341-6ae2ea4aa280', 'model': 'deepseek-v4-flash', 'model_name': 'deepseek-v4-flash', 'stream': False, 'extra_body': {'thinking': {'type': 'disabled'}}, '_type': 'chat-deepseek', 'checkpoint_ns': 'model:68fd80eb-3be4-32bb-a341-6ae2ea4aa280', 'ls_provider': 'deepseek', 'ls_model_name': 'deepseek-v4-flash', 'ls_model_type': 'chat', 'ls_temperature': None})
node: model
content: []

messages
data: (AIMessageChunk(content='Let', additional_kwargs={}, response_metadata={'model_provider': 'deepseek'}, id='lc_run--019dc9f3-64b7-7be1-8deb-92430

#### stream_mode="custom"

将流模式作为列表传入来指定多种流模式

In [ ]:
from langgraph.config import get_stream_writer

def get_weather_with_stream_writer(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"

agent_with_stream_writer = create_agent(
    model=init_chat_model(**LLM_FAST),
    tools=[get_weather_with_stream_writer],
)

for chunk in agent_with_stream_writer.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["updates", "custom"],
    version="v2",
):
    print(f"stream_mode: {chunk['type']}")
    print(f"content: {chunk['data']}\n")

stream_mode: updates
content: {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 284, 'total_tokens': 333, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 284}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': '7d0aadae-bc0f-4028-af4c-e6b435fc7346', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dc9f4-85f6-74c3-85b0-fb90c943d9cb-0', tool_calls=[{'name': 'get_weather_with_stream_writer', 'args': {'city': 'San Francisco'}, 'id': 'call_00_hMlauLqIP6gyiwdYapv5pSH8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 284, 'output_tokens': 49, 'total_tokens': 333, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})]}}

s

#### reasoning

In [15]:
# TODO: DeepSeek 模型的思考模式中，如果进行了工具调用，在后续所有请求中，必须完整回传 reasoning_content 给 API。

#### stream_mode=["messages", "updates"]

同时流式传输作为工具调用生成的部分 JSON 和已完成并解析后执行的工具调用

In [16]:
from langchain.messages import AIMessage, AIMessageChunk, AnyMessage, ToolMessage


def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)
    # N.B. all content is available through token.content_blocks


def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Tool calls: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Tool response: {message.content_blocks}")


for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in Boston?"}]},
    stream_mode=["messages", "updates"],
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
    elif chunk["type"] == "updates":
        for source, update in chunk["data"].items():
            if source in ("model", "tools"):  # `source` captures node name
                _render_completed_message(update["messages"][-1])

Let| me| check| the| weather| in| Boston|.|[{'name': 'get_weather', 'args': '', 'id': 'call_00_SUxK8AffWeM1eVNIbrvSJyj4', 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '{', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '"', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': 'city', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '"', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': ': ', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '"', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': 'Boston', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '"', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '}', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
Tool calls: [{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'call_00_SUxK8AffWeM1eVNIbrvS